In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook explores four performance characteristics of a
# continuous-time low-pass Butterworth filter:
#
#       1. Group delay τg(ω)
#       2. Loss characteristic A(ω)
#       3. Selectivity Fs
#       4. Spectral performance factor Sαβ
#
# Four parameters can be varied:
#
#       N   : filter order
#       ωc  : cutoff angular frequency
#       α   : lower attenuation level used for Sαβ
#       β   : higher attenuation level used for Sαβ
#
# The Butterworth filter is constructed directly with
#
#       scipy.signal.butter(..., analog=True)
#
# Group delay:
#
#       τg(ω) = -dφ(ω)/dω
#
# Loss characteristic:
#
#       A(ω) = 10 log10[1 + (ω/ωc)^(2N)]
#
# Selectivity:
#
#       Fs = N / (2√2 ωc)
#
# For an attenuation level R:
#
#       ωR = ωc [10^(R/10) - 1]^(1/(2N))
#
# and therefore
#
#       Sαβ = BWβ / BWα
#            = [(10^(β/10)-1)/(10^(α/10)-1)]^(1/(2N))
#
# The lower two graphs show Fs and Sαβ as functions of filter order.
# A filled point identifies the current value selected by the N slider.
#
# All curves are updated in real time without recreating the figures.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 7px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:760px;
    max-width:760px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Explore the group delay, loss characteristic, selectivity and spectral
performance factor of a continuous-time Butterworth low-pass filter.
<br>
<b>Interpretation:</b>
The sliders control N, ω<sub>c</sub> and the attenuation levels α and β.
Increasing N produces a sharper loss characteristic and greater selectivity,
while S<sub>α</sub><sup>β</sup> approaches its ideal value 1. The group delay
is not constant and changes significantly around the transition region.
</div>
""", layout=Layout(width='770px', max_width='770px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='250px')
style_opts = {'description_width':'75px'}

order_slider = IntSlider(min=1, max=10, step=1, value=4, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)
wc_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=1.0, description='ωc:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)
alpha_slider = FloatSlider(min=1.0, max=10.0, step=0.5, value=3.0, description='α (dB):', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)
beta_slider = FloatSlider(min=20.0, max=80.0, step=1.0, value=40.0, description='β (dB):', continuous_update=True, readout=True, readout_format='.0f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='270px', max_width='270px'))

# ==============================================================================
# FIGURE 1: GROUP DELAY
# ==============================================================================

fig_gd, ax_gd = plt.subplots(figsize=(4.6, 3.0))

gd_line, = ax_gd.plot([], [], 'r-', linewidth=2.0, label='τg(ω)')
wc_gd_line = ax_gd.axvline(1.0, color='black', linestyle=':', linewidth=1.1, label='ωc')
zero_gd_line = ax_gd.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_gd.set_xlabel('Angular Frequency ω (rad/s)', fontsize=9)
ax_gd.set_ylabel('Group Delay τg(ω) (s)', fontsize=9)
ax_gd.set_title('Group Delay', fontsize=11, fontweight='bold', pad=5)
ax_gd.tick_params(axis='both', labelsize=8)
ax_gd.grid(True, linestyle=':', alpha=0.5)
ax_gd.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, fontsize=7)
ax_gd.set_xlim(0.0, 10.0)
ax_gd.set_ylim(0.0, 30.0)

fig_gd.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_gd.canvas.header_visible = False
fig_gd.canvas.toolbar_visible = False
fig_gd.canvas.resizable = False
fig_gd.canvas.layout.width = '460px'
fig_gd.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 2: LOSS CHARACTERISTIC
# ==============================================================================

fig_loss, ax_loss = plt.subplots(figsize=(4.6, 3.0))

loss_line, = ax_loss.plot([], [], 'r-', linewidth=2.0, label='A(ω)')
wc_loss_line = ax_loss.axvline(1.0, color='black', linestyle=':', linewidth=1.1, label='ωc')
loss3_line = ax_loss.axhline(3.0103, color='gray', linestyle='--', linewidth=0.9, label='3.01 dB')

ax_loss.set_xlabel('Angular Frequency ω (rad/s)', fontsize=9)
ax_loss.set_ylabel('Loss A(ω) (dB)', fontsize=9)
ax_loss.set_title('Loss Characteristic', fontsize=11, fontweight='bold', pad=5)
ax_loss.tick_params(axis='both', labelsize=8)
ax_loss.grid(True, linestyle=':', alpha=0.5)
ax_loss.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=7)
ax_loss.set_xlim(0.0, 10.0)
ax_loss.set_ylim(0.0, 100.0)

fig_loss.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_loss.canvas.header_visible = False
fig_loss.canvas.toolbar_visible = False
fig_loss.canvas.resizable = False
fig_loss.canvas.layout.width = '460px'
fig_loss.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 3: SELECTIVITY
# ==============================================================================

fig_sel, ax_sel = plt.subplots(figsize=(4.6, 3.0))

sel_line, = ax_sel.plot([], [], 'r-', linewidth=2.0, label='Fs(N)')
sel_point, = ax_sel.plot([], [], 'ro', markersize=6, label='Current N')

ax_sel.set_xlabel('Filter Order N', fontsize=9)
ax_sel.set_ylabel('Selectivity Fs', fontsize=9)
ax_sel.set_title('Selectivity', fontsize=11, fontweight='bold', pad=5)
ax_sel.tick_params(axis='both', labelsize=8)
ax_sel.grid(True, linestyle=':', alpha=0.5)
ax_sel.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, fontsize=7)
ax_sel.set_xlim(1.0, 10.0)
ax_sel.set_xticks(np.arange(1, 11, 1))
ax_sel.set_ylim(0.0, 8.0)

fig_sel.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_sel.canvas.header_visible = False
fig_sel.canvas.toolbar_visible = False
fig_sel.canvas.resizable = False
fig_sel.canvas.layout.width = '460px'
fig_sel.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 4: SPECTRAL PERFORMANCE FACTOR
# ==============================================================================

fig_sp, ax_sp = plt.subplots(figsize=(4.6, 3.0))

sp_line, = ax_sp.plot([], [], 'r-', linewidth=2.0, label='Sαβ(N)')
sp_point, = ax_sp.plot([], [], 'ro', markersize=6, label='Current N')
ideal_line = ax_sp.axhline(1.0, color='gray', linestyle='--', linewidth=0.9, label='Ideal = 1')

ax_sp.set_xlabel('Filter Order N', fontsize=9)
ax_sp.set_ylabel('Spectral Factor Sαβ', fontsize=9)
ax_sp.set_title('Spectral Performance Factor', fontsize=11, fontweight='bold', pad=5)
ax_sp.tick_params(axis='both', labelsize=8)
ax_sp.grid(True, linestyle=':', alpha=0.5)
ax_sp.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=7)
ax_sp.set_xlim(1.0, 10.0)
ax_sp.set_xticks(np.arange(1, 11, 1))
ax_sp.set_ylim(1.0, 20.0)

fig_sp.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_sp.canvas.header_visible = False
fig_sp.canvas.toolbar_visible = False
fig_sp.canvas.resizable = False
fig_sp.canvas.layout.width = '460px'
fig_sp.canvas.layout.height = '305px'

# ==============================================================================
# FIXED FREQUENCY AXIS
# ==============================================================================

omega = np.linspace(0.001, 10.0, 3000)
N_values = np.arange(1, 11)

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_butterworth_performance(change=None):

    N = order_slider.value
    wc = wc_slider.value
    alpha = alpha_slider.value
    beta = beta_slider.value

    # --------------------------------------------------------------------------
    # BUTTERWORTH TRANSFER FUNCTION
    # --------------------------------------------------------------------------

    b, a = signal.butter(N, wc, btype='low', analog=True, output='ba')

    # --------------------------------------------------------------------------
    # FREQUENCY RESPONSE
    # --------------------------------------------------------------------------

    _, H = signal.freqs(b, a, worN=omega)

    phase = np.unwrap(np.angle(H))

    # --------------------------------------------------------------------------
    # GROUP DELAY
    #
    # τg(ω) = -dφ/dω
    # --------------------------------------------------------------------------

    group_delay = -np.gradient(phase, omega)

    # --------------------------------------------------------------------------
    # LOSS CHARACTERISTIC
    #
    # A(ω) = 10 log10[1 + (ω/ωc)^(2N)]
    # --------------------------------------------------------------------------

    loss = 10.0 * np.log10(1.0 + (omega / wc)**(2 * N))

    # --------------------------------------------------------------------------
    # SELECTIVITY
    #
    # Fs = N / (2√2 ωc)
    # --------------------------------------------------------------------------

    Fs = N / (2.0 * np.sqrt(2.0) * wc)
    Fs_values = N_values / (2.0 * np.sqrt(2.0) * wc)

    # --------------------------------------------------------------------------
    # SPECTRAL PERFORMANCE FACTOR
    #
    # ωR = ωc [10^(R/10)-1]^(1/(2N))
    #
    # Sαβ = BWβ / BWα
    # --------------------------------------------------------------------------

    BW_alpha = wc * (10.0**(alpha / 10.0) - 1.0)**(1.0 / (2.0 * N))
    BW_beta = wc * (10.0**(beta / 10.0) - 1.0)**(1.0 / (2.0 * N))
    S = BW_beta / BW_alpha

    S_values = ((10.0**(beta / 10.0) - 1.0) / (10.0**(alpha / 10.0) - 1.0))**(1.0 / (2.0 * N_values))

    # --------------------------------------------------------------------------
    # UPDATE GROUP DELAY
    # --------------------------------------------------------------------------

    gd_line.set_data(omega, group_delay)
    wc_gd_line.set_xdata([wc, wc])

    ax_gd.set_xlim(0.0, 10.0)
    ax_gd.set_ylim(0.0, 30.0)

    # --------------------------------------------------------------------------
    # UPDATE LOSS CHARACTERISTIC
    # --------------------------------------------------------------------------

    loss_line.set_data(omega, loss)
    wc_loss_line.set_xdata([wc, wc])

    ax_loss.set_xlim(0.0, 10.0)
    ax_loss.set_ylim(0.0, 100.0)

    # --------------------------------------------------------------------------
    # UPDATE SELECTIVITY
    # --------------------------------------------------------------------------

    sel_line.set_data(N_values, Fs_values)
    sel_point.set_data([N], [Fs])

    ax_sel.set_xlim(1.0, 10.0)
    ax_sel.set_ylim(0.0, 8.0)

    # --------------------------------------------------------------------------
    # UPDATE SPECTRAL PERFORMANCE FACTOR
    # --------------------------------------------------------------------------

    sp_line.set_data(N_values, S_values)
    sp_point.set_data([N], [S])

    ax_sp.set_xlim(1.0, 10.0)
    ax_sp.set_ylim(1.0, 20.0)

    # --------------------------------------------------------------------------
    # INFORMATION PANEL
    # --------------------------------------------------------------------------

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 9px;
        margin-top:9px;
        font-size:12px;
        line-height:1.60;
        background:white;
        width:265px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Butterworth low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Cutoff frequency:</b>
        <span style="color:#0066cc;">ωc = {wc:.2f} rad/s</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Performance quantities:</b>
    </div>

    <div>
        <b>Selectivity:</b>
        <span style="color:#0066cc;">Fs = {Fs:.6f}</span>
    </div>

    <div>
        <b>Attenuation levels:</b>
        <span style="color:#0066cc;">α = {alpha:.1f} dB, β = {beta:.1f} dB</span>
    </div>

    <div>
        <b>BWα:</b>
        <span style="color:#0066cc;">{BW_alpha:.6f} rad/s</span>
    </div>

    <div>
        <b>BWβ:</b>
        <span style="color:#0066cc;">{BW_beta:.6f} rad/s</span>
    </div>

    <div>
        <b>Spectral factor:</b>
        <span style="color:#0066cc;">Sαβ = {S:.6f}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        Increasing N steepens the loss characteristic and increases
        selectivity. The spectral performance factor approaches its ideal
        value 1, while the group delay becomes increasingly frequency
        dependent around the transition region.
    </div>

    </div>
    """

    # --------------------------------------------------------------------------
    # REDRAW EXISTING FIGURES ONLY
    # --------------------------------------------------------------------------

    fig_gd.canvas.draw_idle()
    fig_loss.canvas.draw_idle()
    fig_sel.canvas.draw_idle()
    fig_sp.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_butterworth_performance, names='value')
wc_slider.observe(update_butterworth_performance, names='value')
alpha_slider.observe(update_butterworth_performance, names='value')
beta_slider.observe(update_butterworth_performance, names='value')

# ==============================================================================
# LAYOUT: 2 x 2 FIGURE GRID
# ==============================================================================

controls = VBox([parameter_title, order_slider, wc_slider, alpha_slider, beta_slider, info_html], layout=Layout(width='280px', min_width='280px', max_width='280px', flex='0 0 280px', align_items='flex-start'))

top_row = HBox([fig_gd.canvas, fig_loss.canvas], layout=Layout(width='930px', align_items='flex-start', justify_content='flex-start'))

bottom_row = HBox([fig_sel.canvas, fig_sp.canvas], layout=Layout(width='930px', align_items='flex-start', justify_content='flex-start'))

plot_grid = VBox([top_row, bottom_row], layout=Layout(width='930px', align_items='flex-start', justify_content='flex-start'))

main_layout = HBox([controls, plot_grid], layout=Layout(width='1210px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE DATA
# ==============================================================================

update_butterworth_performance()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)
display(main_layout)